In [78]:
import numpy as np
import torch
import pyro
import pyro.distributions as dist
from scipy.integrate import dblquad
from scipy.stats import multivariate_normal
import numpyro
import matplotlib.pyplot as plt
from scipy.special import logsumexp

In [2]:
def pyro_model_(data, parameter_len, epsilon, prior_sigma):
    prior_parameter = pyro.sample("prior_parameter", dist.MultivariateNormal(torch.zeros(parameter_len), torch.eye(parameter_len)*prior_sigma))
    mean = torch.tensor([prior_parameter[0] - torch.max(prior_parameter), prior_parameter[1]]).float()
    with pyro.plate("data_plate"):
        pyro.sample("obs", dist.MultivariateNormal(mean, (epsilon**2)*torch.eye(len(data))), obs=data)    


def run_HMC(data, parameter_len, epsilon, prior_sigma, stepsize, num_samples, initial_params, warmup_steps):

    pyro.clear_param_store()
    pyro_model = lambda data: pyro_model_(data=data, parameter_len=parameter_len, epsilon=epsilon, prior_sigma=prior_sigma)
    pyro_kernel =  pyro.infer.mcmc.HMC(model=pyro_model, step_size=stepsize)
    pyro_mcmc = pyro.infer.mcmc.MCMC(kernel=pyro_kernel, num_samples=num_samples, initial_params={'prior_parameter': initial_params}, warmup_steps=warmup_steps)
    pyro_mcmc.run(data)
    samples = pyro_mcmc.get_samples()["prior_parameter"]
                    
    return samples

data = torch.tensor([-1.,-1.])
parameter_len = 2
prior_sigma = 1.
eps = 0.1




In [3]:
def ABC_prob(epsilon, prior_sigma, data):
    def fn_(y, x, epsilon, prior_sigma):
        #y is theta2, x is theta1
        lh = multivariate_normal.pdf(data.numpy(), mean=np.array([x - np.maximum(x,y), y]), cov=epsilon**2)
        pr = multivariate_normal.pdf(np.array([x,y]), mean=np.array([0.,0.]), cov=prior_sigma**2)
        return lh*pr
    fn = lambda y, x: fn_(y, x, epsilon=epsilon, prior_sigma=prior_sigma)
    numerator = dblquad(fn, -np.inf, np.inf, lambda x:x, np.inf)[0]
    denominator = dblquad(fn, -np.inf, np.inf, -np.inf, np.inf)[0]
    return numerator / denominator

In [11]:
def GR(x):
    return numpyro.diagnostics.gelman_rubin(x)

def ABC_error(p_eps):
    return (1-p_eps)**2

def MCMC_error(p_eps, weights, particles):
    return (p_eps - np.sum([weights[i] for i in range(len(weights)) if particles[i][1] > particles[i][0]]))**2

def Overall_error(weights, particles):
    return (1 - np.sum([weights[i] for i in range(len(weights)) if particles[i][1] > particles[i][0]]))**2


In [181]:
# initial_eps = 1.
# num_particles = 20
# eps_range = [0.1,0.3,0.5,0.7,0.9]

# initial_warmup_steps = 1000
# resample_rate = 100
# initial_num_samples = num_particles*resample_rate
# initial_params = torch.tensor([0.,0.])
# stepsize = 1.

# warmup_steps = 20
# num_samples = 100

#testing parameters
num_particles = 2
eps_range = [0.1,0.3]
initial_warmup_steps = 1
resample_rate = 1
initial_num_samples = num_particles*resample_rate
initial_params = torch.tensor([0.,0.])
stepsize = 0.1

warmup_steps = 2
num_samples = 5



p0_samples = run_HMC(data=data, 
        parameter_len=parameter_len, 
        epsilon=initial_eps, 
        prior_sigma=prior_sigma, 
        stepsize=stepsize, 
        num_samples=initial_num_samples, 
        initial_params=initial_params, 
        warmup_steps=initial_warmup_steps).numpy()[::resample_rate]



#SMC
all_particles_full = []
all_particles = []
all_weights = []
all_gr = []
all_abcerror = []
all_mcmcerror = []
all_overallerror = []

for eps in eps_range:
    logweights = np.zeros(num_particles)
    particles = np.zeros([num_particles,parameter_len])
    particles_full = np.zeros([num_particles, num_samples, parameter_len])
    for n in range(num_particles):
        print("Working on eps {}, particle, {}...".format(eps, n))
        #smc
        sample = p0_samples[n]
        wn = -1/(2*eps**2)*((data.numpy()[0]-(sample[0]-np.max(sample)))**2+(data.numpy()[1]-sample[1])**2)
        logweights[n] = wn
        
        #MCMC
        pn = run_HMC(data=data, 
                parameter_len=parameter_len, 
                epsilon=eps, 
                prior_sigma=prior_sigma, 
                stepsize=stepsize, 
                num_samples=num_samples, 
                initial_params=torch.tensor(p0_samples[n]), 
                warmup_steps=warmup_steps).numpy()
        particles[n] = pn[-1]
        particles_full[n] = pn
    
    weights = np.exp(logweights - logsumexp(logweights))
    
    all_particles_full.append(particles_full)
    all_particles.append(particles)
    all_weights.append(weights)
    
    
    #Metrics
    gr = np.array([GR(np.array(particles_full)[:,:,0]),GR(np.array(particles_full)[:,:,1])])
    abcerror = ABC_error(eps)
    mcmcerror = MCMC_error(eps, weights, particles)
    overallerror = Overall_error(weights, particles)
    
    all_gr.append(gr)
    all_abcerror.append(abcerror)
    all_mcmcerror.append(mcmcerror)
    all_overallerror.append(overallerror)
    
#     np.savez("../abcerrorresults/eps{}.npz".format(str(eps).replace(".","-")),
#         particles_full=particles_full, 
#              particles=particles, 
#              weights=weights,
#             gr=gr,
#             abcerror=abcerror,
#             mcmcerror=mcmcerror,
#             overallerror=overallerror)
    

Sample: 100%|█████████████████████████████████████████████| 3/3 [00:00, 47.62it/s, step size=1.60e+00, acc. prob=0.672]


Working on eps 0.1, particle, 0...


Sample: 100%|█████████████████████████████████████████████| 7/7 [00:00, 59.32it/s, step size=2.88e+00, acc. prob=0.000]


Working on eps 0.1, particle, 1...


Sample: 100%|█████████████████████████████████████████████| 7/7 [00:00, 19.72it/s, step size=7.19e-01, acc. prob=0.600]


Working on eps 0.3, particle, 0...


Sample: 100%|█████████████████████████████████████████████| 7/7 [00:00, 39.43it/s, step size=2.88e+00, acc. prob=0.000]


Working on eps 0.3, particle, 1...


Sample: 100%|█████████████████████████████████████████████| 7/7 [00:00, 10.72it/s, step size=7.19e-01, acc. prob=0.273]


In [ ]:
plt.plot(eps_range, eps_res)